In [77]:
from adam_assist import ASSISTPropagator
from mpcq import MPCObservations
from mpcq.orbits import MPCOrbits
import pyarrow.compute as pc
import numpy as np

from adam_core.coordinates import CoordinateCovariances, SphericalCoordinates
from adam_core.coordinates.origin import Origin
from adam_core.observers import Observers
from adam_core.orbit_determination.evaluate import OrbitDeterminationObservations
from adam_core.coordinates.residuals import Residuals

In [78]:
all_observations = MPCObservations.from_parquet("../observations.parquet")
all_orbits = MPCOrbits.from_parquet("../mpc_orbits.parquet")

In [79]:
# Pull only our object and only observations with RMS. Should get 18 observations
object_id = "1999 RQ36"

observations = MPCObservations.from_pyarrow(all_observations.table.filter(pc.equal(all_observations.requested_provid, object_id)))
have_ra = ~np.isnan(observations.rmsra)
have_dec = ~np.isnan(observations.rmsdec)
observations = MPCObservations.from_pyarrow(observations.table.filter(have_ra & have_dec))
station_counts = pc.value_counts(observations.stn)
print(f"Observation count {len(observations)}, per station {station_counts.to_pylist()}")

orbits = MPCOrbits.from_pyarrow(all_orbits.table.filter(pc.equal(all_orbits.requested_provid, object_id)))
print(f"Orbits {orbits}")

Observation count 18, per station [{'values': 'Y05', 'counts': 4}, {'values': '691', 'counts': 5}, {'values': '291', 'counts': 3}, {'values': '033', 'counts': 6}]
Orbits MPCOrbits(size=1)


In [80]:
# Make observations with covariances
obs_time = observations.obstime
codes = observations.stn

# `mpcq`'s `MPCObservations` includes uncertainty columns:
# - rmsra, rmsdec, rmscorr (and rmsmag)
#
# These RMS values come from the MPC database and are in arcseconds. By ADES/MPC convention,
# `rmsra` is RA uncertainty *cos(dec). We convert into degrees and back out RA sigma
# (so that downstream `Residuals` can apply its own cos(latitude) scaling consistently).
dec_deg = observations.dec.to_numpy(zero_copy_only=False)
cos_dec = np.cos(np.deg2rad(dec_deg))

sigma_ra_cosdec_deg = observations.rmsra.to_numpy(zero_copy_only=False) / 3600.0
sigma_dec_deg = observations.rmsdec.to_numpy(zero_copy_only=False) / 3600.0
sigma_ra_deg = np.where(
    np.isfinite(cos_dec) & (cos_dec != 0.0),
    sigma_ra_cosdec_deg / cos_dec,
    np.nan,
)

# Include RA/Dec correlation if present; treat missing correlation as 0 (uncorrelated).
corr = observations.rmscorr.to_numpy(zero_copy_only=False)
corr = np.where(np.isfinite(corr), corr, 0.0)

N = len(observations)
cov = np.full((N, 6, 6), np.nan, dtype=np.float64) # use 0, not np.nan. OD doesn't like nan
#cov[:, :, 1] = 0
#cov[:, :, 2] = 0
cov[:, 1, 1] = np.nan_to_num(sigma_ra_deg**2, nan=1.0e-9)
cov[:, 2, 2] = np.nan_to_num(sigma_dec_deg**2, nan=1.0e-9)
cov[:, 1, 2] = np.nan_to_num(corr * sigma_ra_deg * sigma_dec_deg)
cov[:, 2, 1] = cov[:, 1, 2]

print(cov[2, :, :])
print(cov[3, :, :])
# print(np.sum(np.isnan(cov), axis=0))

coords = SphericalCoordinates.from_kwargs(
    lon=observations.ra.to_numpy(zero_copy_only=False),
    lat=observations.dec.to_numpy(zero_copy_only=False),
    time=obs_time,
    origin=Origin.from_kwargs(code=codes),
    frame="equatorial",
    covariance=CoordinateCovariances.from_matrix(cov),
)

observers = Observers.from_codes(codes=codes, times=obs_time)

od_observations = OrbitDeterminationObservations.from_kwargs(
    id=observations.obsid.to_numpy(zero_copy_only=False),
    coordinates=coords,
    observers=observers,
)

[[           nan            nan            nan            nan
             nan            nan]
 [           nan 2.37699344e-09 0.00000000e+00            nan
             nan            nan]
 [           nan 0.00000000e+00 1.73611111e-09            nan
             nan            nan]
 [           nan            nan            nan            nan
             nan            nan]
 [           nan            nan            nan            nan
             nan            nan]
 [           nan            nan            nan            nan
             nan            nan]]
[[           nan            nan            nan            nan
             nan            nan]
 [           nan 2.72813379e-09 0.00000000e+00            nan
             nan            nan]
 [           nan 0.00000000e+00 1.51234568e-09            nan
             nan            nan]
 [           nan            nan            nan            nan
             nan            nan]
 [           nan            nan            nan   

In [81]:
# Existing Gauss IOD. Tends to show junk
from adam_core.orbit_determination.iod import iod

# Initial orbit determination from observations only
fitted_orbits, fitted_members = iod(
    od_observations,
    min_obs=6,
    # min_arc_length=0.001, # 1.0,
    rchi2_threshold=1e14, # 1_000,
    contamination_percentage=50, # ???
    observation_selection_method="combinations",
    iterate=False,
    light_time=True,
    propagator=ASSISTPropagator,
)

print(fitted_orbits)
print(fitted_members.residuals)

== IN IOD
Processable A True, num obs 18, min_obs 6, contamination_percentage 50
Max outliers 9
selected_index shape (816, 3)
obs_ids count before taking head 816
obs_ids [['Lj6D0X1y0000G63j010000001' 'Llv3RlEA0000GVeX010000001'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'Llv3RlEA0000GVeX010000002'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmN2Xl5S0000GaZx010000001'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmN2Xl5S0000GaZx010000002'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmN2Xl5S0000GaZx010000003'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmQ3c5DS0000GbBX010000001'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmQ3c5DS0000GbBX010000002'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LmQ3c5DS0000GbBX010000003'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001' 'LnVIJU2K0000Glt3010000001'
  'LnwJLx450000GpgX010000003']
 ['Lj6D0X1y0000G63j010000001

In [82]:
print(fitted_orbits.coordinates.x[0].as_py(),
      fitted_orbits.coordinates.y[0].as_py(),
      fitted_orbits.coordinates.z[0].as_py(),
      fitted_orbits.coordinates.vx[0].as_py(),
      fitted_orbits.coordinates.vy[0].as_py(),
      fitted_orbits.coordinates.vz[0].as_py(),
      fitted_orbits.coordinates.time.days[0].as_py(),
      fitted_orbits.coordinates.time.nanos[0].as_py(),
      fitted_orbits.coordinates.frame)
def print_kepl(kepl, idx=0):
    getv = lambda name : kepl.table.column(name)[idx].as_py()
    print(f"a={getv('a')} e={getv('e')} i={getv('i')} raan={getv('raan')} ap={getv('ap')} time={getv('time')} origin={getv('origin')}")
print_kepl(fitted_orbits.coordinates.to_keplerian())
print(fitted_orbits.coordinates.frame)

-0.7744095305266402 -2.47295832281171 -0.6140821446704292 0.004602741083756447 -0.005586946552969135 0.0012876721579375052 60497 85475218438287 ecliptic
a=1.759596226150504 e=0.6599945086496583 i=23.601180190069286 raan=285.4575032264362 ap=163.00501148337824 time={'days': 60497, 'nanos': 85475218438287} origin={'code': 'SUN'}
ecliptic


In [83]:
# Sanity check against MCP orbit
ephemeris = ASSISTPropagator().generate_ephemeris(orbits.orbits(), observers, chunk_size=1, max_processes=1)
residuals = Residuals.calculate(coords, ephemeris.coordinates)
chi2 = residuals.chi2.to_numpy()

print(residuals)
print(chi2)
chi2_total = np.sum(chi2)
rchi2 = chi2_total / (2 * len(chi2) - 6)
print(f"chi2_total={chi2_total}, rchi2={rchi2}")

Residuals(size=18)
[ 46.54763282  32.55784645  24.95498971  11.89545164  36.06822225
  22.18794267  57.21110121  13.75814708  58.07848653  14.61783809
  35.49458046 141.2141632    4.2521389    9.73147453  25.33648142
  18.05024008   0.43032513  29.43370257]
chi2_total=581.8207647433273, rchi2=19.394025491444246


In [84]:
def print_mpc_orbit(orbs_set, idx=0):
    getv = lambda name : orbs_set.table.column(name)[idx].as_py()
    print(f"MPC orbit q={getv('q')} e={getv('e')} i={getv('i')} ap={getv('argperi')} tp={getv('peri_time')} epoch={getv('epoch')}")
print_mpc_orbit(orbits)

MPC orbit q=0.896590640193009 e=0.20373108132691 i=6.0329039801119 ap=66.3943776911573 tp=60675.7378717287 epoch={'days': 60800, 'nanos': 0}


In [85]:
# print(ephemeris.coordinates.time.days, ephemeris.coordinates.time.nanos)
# element 2 is 37 min off fitted orbit, but in a different frame (observer)
el = ephemeris.coordinates[2].to_cartesian()
print(el.x[0].as_py(),
      el.y[0].as_py(),
      el.z[0].as_py(),
      el.vx[0].as_py(),
      el.vy[0].as_py(),
      el.vz[0].as_py(),
      el.time.days[0].as_py(),
      el.time.nanos[0].as_py(), el.frame)

-0.23605860904739764 -0.2554999815546043 -0.26302110679073043 -0.003113304373681602 -0.0024598612146256643 -0.001010106244787597 60497 83260500000000 equatorial


In [86]:
# from https://github.com/B612-Asteroid-Institute/adam_impacts_study/blob/main/src/adam_impact_study/fo_od.py

from adam_fo import fo
import pyarrow as pa
from adam_core.observations.ades import (
    ADES_to_string,
    ADESObservations,
    ObsContext,
    ObservatoryObsContext,
    SubmitterObsContext,
    TelescopeObsContext,
)


def observations_to_ades(observations: MPCObservations): # -> Tuple[str, ADESObservations]:
    """
    Convert Observations to ADES format string.

    Parameters
    ----------
    observations : Observations
        Observations to convert

    Returns
    -------
    Tuple[str, ADESObservations]
        Tuple containing:
        - ADES format string
        - ADESObservations object
    """
    # Extract the original orbit_id since trkSub has an 8-character limit
    orbit_ids = observations.requested_provid #   orbit_ids

    # Convert uncertainties in RA and Dec to arcseconds
    # and adjust the RA uncertainty to account for the cosine of the declination
    # sigma_ra_cos_dec = (
    #     np.cos(np.radians(observations.coordinates.lat.to_numpy(zero_copy_only=False)))
    #     * observations.coordinates.covariance.sigmas[:, 1]
    # )
    # sigma_ra_cos_dec_arcseconds = pa.array(sigma_ra_cos_dec * 3600, type=pa.float64())
    # sigma_dec_arcseconds = pa.array(
    #     observations.coordinates.covariance.sigmas[:, 2] * 3600, type=pa.float64()
    # )

    # # Replace nans with nulls using pyarrow
    # sigma_ra_cos_dec_arcseconds = pc.if_else(
    #     pc.is_nan(sigma_ra_cos_dec_arcseconds), None, sigma_ra_cos_dec_arcseconds
    # )
    # sigma_dec_arcseconds = pc.if_else(
    #     pc.is_nan(sigma_dec_arcseconds), None, sigma_dec_arcseconds
    # )

    # Serialize observations to an ADES table
    ades_observations = ADESObservations.from_kwargs(
        trkSub=pc.utf8_slice_codeunits(orbit_ids, 0, 8),
        obsTime=observations.obstime, # coordinates.time,
        ra=observations.ra, #  observations.coordinates.lon,
        dec=observations.dec, # observations.coordinates.lat,
        # rmsRACosDec=observations.rmsra, #sigma_ra_cos_dec_arcseconds,
        # rmsDec=observations.rmsdec, #sigma_dec_arcseconds,
        # mag=observations.photometry.mag,
        # rmsMag=observations.photometry.mag_sigma,
        # band=observations.photometry.filter,
        stn=observations.stn, #pa.repeat("X05", len(observations)),
        mode=pa.repeat("NA", len(observations)),
        astCat=pa.repeat("NA", len(observations)),
    )

    # Minimal obscontext representing our simulated X05 survey
    # obs_contexts = {
    #     "X05": ObsContext(
    #         observatory=ObservatoryObsContext(
    #             mpcCode="X05", name="Vera C. Rubin Observatory - LSST"
    #         ),
    #         submitter=SubmitterObsContext(
    #             name="K. Kiker",
    #             institution="B612 Asteroid Institute",
    #         ),
    #         observers=["J. Doe"],
    #         measurers=["J. Doe"],
    #         telescope=TelescopeObsContext(
    #             name="Simonyi Survey Telescope",
    #             design="Reflector",
    #         ),
    #     ),
    # }

    codes = [v.as_py() for v in pc.unique(observations.stn)]
    print(codes)

    telescope = TelescopeObsContext(
                name="Thing",
                design="Reflector",
                detector="CCD", 
                aperture=40.0,
            )

    obs_contexts = {
        name: ObsContext(
            observatory=ObservatoryObsContext(
                mpcCode=name, name=f"obs {name}"
            ),
            submitter=SubmitterObsContext(
                name="K. Kiker",
                institution="B612 Asteroid Institute",
            ),
            observers=["J. Doe"],
            measurers=["J. Doe"],
            telescope=telescope,
        ) for name in codes
    }
    
    ades_string = ADES_to_string(ades_observations, obs_contexts)
    return ades_string, ades_observations


def run_fo_od(
    observations: MPCObservations,
    fo_result_dir: str
): # -> Tuple[Orbits, ADESObservations, Optional[str]]:
    """Run Find_Orb orbit determination with directory-based paths

    Parameters
    ----------
    observations : Observations
        Observations to process
    fo_result_dir : str
        Directory where Find_Orb output files will be written

    Returns
    -------
    Tuple[Orbits, ADESObservations, Optional[str]]
        Tuple containing:
        - Determined orbit
        - Processed observations
        - Error message (if any)
    """
    # This function is only valid for a single orbit_id
    if len(observations.requested_provid.unique()) > 1:
        raise ValueError("This function is only valid for a single orbit_id")

    ades_string, _ = observations_to_ades(observations)

    orbit, rejected, error = fo(
        ades_string,
        out_dir=fo_result_dir,
        # Ignore clean_up in favor of keeping whitelisted files below
        clean_up=False,
    )
    print(orbit, type(orbit))
    print(rejected)
    print(error)
    return orbit, rejected

fo_orbit, fo_rejected = run_fo_od(observations, "../fo_dir")

['Y05', '691', '291', '033']
Orbits(size=1) <class 'adam_core.orbits.orbits.Orbits'>
ADESObservations(size=12)
None


In [87]:
ephemeris = ASSISTPropagator().generate_ephemeris(fo_orbit, observers, chunk_size=1, max_processes=1)
residuals = Residuals.calculate(coords, ephemeris.coordinates)
chi2 = residuals.chi2.to_numpy()

print(residuals)
print(chi2)
chi2_total = np.sum(chi2)
rchi2 = chi2_total / (2 * len(chi2) - 6)
print(f"chi2_total={chi2_total}, rchi2={rchi2}")

Residuals(size=18)
[7.97796849e+12 8.23996128e+12 1.20303852e+13 1.08741562e+13
 5.25632034e+12 1.45221647e+13 3.39634871e+12 2.24581118e+12
 5.66550216e+12 1.54383904e+13 1.54829860e+13 6.84873638e+12
 1.31583065e+12 1.85132872e+12 1.95337070e+12 8.37958872e+11
 3.46597768e+11 5.80742515e+11]
chi2_total=114864560308986.47, rchi2=3828818676966.216


In [88]:
print_kepl(fo_orbit.coordinates.to_keplerian())

a=2.4947326832898153 e=0.5602654182445305 i=1.9680188214654295 raan=231.58053185819654 ap=246.44159652210314 time={'days': 60699, 'nanos': 7742390392721} origin={'code': 'SUN'}


In [89]:
print_kepl(fo_orbit.coordinates.to_keplerian())

a=2.4947326832898153 e=0.5602654182445305 i=1.9680188214654295 raan=231.58053185819654 ap=246.44159652210314 time={'days': 60699, 'nanos': 7742390392721} origin={'code': 'SUN'}


In [73]:
[v.to_iso8601()[0].as_py() for v in observations.obstime]

['2024-07-05T23:45:35.200',
 '2024-07-06T00:38:27.100',
 '2024-07-06T23:07:40.500',
 '2024-07-07T00:00:32.300',
 '2024-12-27T01:34:54.620',
 '2024-12-27T02:11:32.640',
 '2025-01-24T02:07:53.180',
 '2025-01-24T02:16:48.860',
 '2025-01-24T02:25:41.950',
 '2025-01-27T02:18:40.320',
 '2025-01-27T02:42:54.430',
 '2025-01-27T03:07:09.410',
 '2025-04-01T19:08:44.730',
 '2025-04-01T19:13:02.200',
 '2025-04-01T19:17:18.810',
 '2025-04-28T20:20:27.450',
 '2025-04-28T20:25:44.540',
 '2025-04-28T20:31:03.360']